# Ovillos — Data Cleaning & Normalization

Analyze and normalize the `ovillos-prod.xlsx` reference data from SIC JAC.
Focus on columns A (codigo), B (descripcion), and L (SUB-CATEGORIA).

**Source:** `docs/reference/warehouse/cleaned/ovillos-prod.xlsx`

In [7]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

df_source = Path("../../reference/warehouse/cleaned/ovillos-prod.xlsx")

---
## 1. Load & Overview

In [8]:
df = pd.read_excel(df_source)
print(f"Shape: {df.shape}")
df.head(10)

Shape: (2132, 12)


,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA
0,01-26-01,KANTUTA 3045 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18
1,01-26-05,AM. LIMON 0024 2/18,KILOS,0.0,0.0,201.5,201.5,201.5,201.5,0.0,0.0,2/18
2,01-26-06,AZUL PETROLEO 7009 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18
3,01-26-11,BLANCO 0010 2/18,KILOS,0.0,0.0,202.5,202.5,202.5,202.5,0.0,0.0,2/18
4,01-26-13,BLANCO 0010 2/18,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/18
5,01-26-16,V. ESMERALDA 4033 2/18,KILOS,0.0,0.0,203.5,203.5,203.5,203.5,0.0,0.0,2/18
6,01-26-18,AZUL PASTEL 7012 2/18,KILOS,0.0,0.0,203.5,203.5,203.5,203.5,0.0,0.0,2/18
7,01-26-19,VERDE 4073 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,2/18
8,01-26-21,CICLAN 2004 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,2/18
9,01-26-23,CICLAN OSCURO 2005 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   codigo            2132 non-null   str    
 1   descripcion       2132 non-null   str    
 2   unidad            2132 non-null   str    
 3   Saldoin-Cantidad  2132 non-null   float64
 4   Saldoin-Valor     2132 non-null   float64
 5   ent-Cantidad      2132 non-null   float64
 6   ent-Valor         2132 non-null   float64
 7   sal-Cantidad      2132 non-null   float64
 8   sal-Valor         2132 non-null   float64
 9   sadout-Cantidad   2132 non-null   float64
 10  saldout-Valor     2132 non-null   float64
 11  SUB-CATEGORIA     2132 non-null   str    
dtypes: float64(8), str(4)
memory usage: 200.0 KB


In [10]:
# Nulls per column
df.isnull().sum()

codigo              0
descripcion         0
unidad              0
Saldoin-Cantidad    0
Saldoin-Valor       0
ent-Cantidad        0
ent-Valor           0
sal-Cantidad        0
sal-Valor           0
sadout-Cantidad     0
saldout-Valor       0
SUB-CATEGORIA       0
dtype: int64

---
## 2. Column A — codigo

Unique identifier per row. Pattern: `XX-YY-ZZ` or `REC-*` for recovered material.

In [11]:
print(f"Unique codes: {df['codigo'].nunique()} out of {len(df)} rows")
print(f"Any nulls: {df['codigo'].isnull().any()}")

# Prefix distribution
codes = df['codigo'].astype(str)
print("\nSample codes:")
print(codes.head(10).to_list())
print(f"\nCodes with 'REC-': {codes.str.startswith('REC-').sum()}")
print(f"Codes with 'MAT-': {codes.str.startswith('MAT-').sum()}")
print(f"Codes with '-SM' suffix: {(codes.str.endswith('-SM').sum())}")
print(f"Codes with '-SM-1' suffix: {(codes.str.endswith('-SM-1').sum())}")
print(f"Codes with '-SN' suffix: {(codes.str.endswith('-SN').sum())}")
print(f"Codes with 'ALP' and 'N' suffix: {(codes.str.contains('ALP') & codes.str.endswith('N')).sum()}")
print(f"Codes with '-SN-1' suffix: {(codes.str.endswith('-SN-1').sum())}")
print(f"Codes with '-SM-N' suffix: {(codes.str.endswith('-SM-N').sum())}")
print(f"Codes with '-CH' suffix: {(codes.str.endswith('-CH').sum())}")
print(f"Codes with '-N' suffix: {codes.str.endswith('-N').sum()}")
print(f"Codes with '-STOLL' suffix: {codes.str.endswith('-STOLL').sum()}")
print(f"Codes longer than 8 chars: {(codes.str.len() > 8).sum()}")
print(f"Codes longer than 11 chars: {(codes.str.len() > 11).sum()}")
print(codes[codes.str.len() > 8].unique())

Unique codes: 2132 out of 2132 rows
Any nulls: False

Sample codes:
['01-26-01', '01-26-05', '01-26-06', '01-26-11', '01-26-13', '01-26-16', '01-26-18', '01-26-19', '01-26-21', '01-26-23']

Codes with 'REC-': 25
Codes with 'MAT-': 14
Codes with '-SM' suffix: 55
Codes with '-SM-1' suffix: 1
Codes with '-SN' suffix: 7
Codes with 'ALP' and 'N' suffix: 1
Codes with '-SN-1' suffix: 0
Codes with '-SM-N' suffix: 1
Codes with '-CH' suffix: 2
Codes with '-N' suffix: 202
Codes with '-STOLL' suffix: 2
Codes longer than 8 chars: 454
Codes longer than 11 chars: 23
<StringArray>
[ '09-32-26-01',  '09-32-26-02',  '09-32-26-03',  '09-32-26-04',
  '09-32-26-05',  '09-32-26-06',  '09-32-26-07',  '09-32-26-08',
  '09-32-26-09',  '09-32-26-11',
 ...
  '57-26-01-SM',  '57-26-02-SM',  '57-26-03-SM',  '57-26-04-SM',
  '57-26-05-SM',  '57-26-06-SM',  '57-26-07-SM',  '57-26-08-SM',
  '57-26-09-SM', 'MAT-REC 2/15']
Length: 454, dtype: str


Create new column `[tipo]` : `HB`, `[estado_mat]` : `empty` and replace `codigo` values to keep only the core identifier (remove prefixes/suffixes). 

In [12]:
df['tipo'] = 'HB'
df['estado_mat'] = ''
df.loc[df['codigo'].astype(str).str.endswith('-N'), 'tipo'] = 'N'
df.loc[df['codigo'].astype(str).str.endswith('-SN' or '-SM-N'), 'tipo'] = 'SM N'
df.loc[df['codigo'].astype(str).str.endswith('-SM'), 'tipo'] = 'SM'
df.loc[df['codigo'].astype(str).str.endswith('-M'), 'tipo'] = 'M'
df.loc[df['codigo'].astype(str).str.endswith('-CH'), 'tipo'] = 'CH'
df.loc[df['codigo'].astype(str).str.endswith('-STOLL'), 'tipo'] = 'STOLL'
df.loc[df['codigo'].astype(str).str.startswith('REC-'), 'estado_mat'] = 'REC'
df.loc[df['codigo'].astype(str).str.startswith('MAT-REC-' or 'MAT-REC '), 'estado_mat'] = 'MAT-REC'
df.loc[df['codigo'].astype(str).str.startswith('ALP-' or 'ALPACA-' or 'ALPACA '), 'tipo'] = 'ALPACA'
df.loc[df['codigo'].astype(str).str.startswith('CC-'), 'tipo'] = 'CICLAN'
df.loc[df['codigo'].astype(str).str.contains('ALP' and 'N'), 'tipo'] = 'ALPACA / SM N'

for suffix in ['-SM', '-SM-N', '-SN', '-CH', '-N', '-M', '-STOLL']:
    mask = df['codigo'].str.endswith(suffix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(suffix + '$', '', regex=True)

for prefix in ['REC-', 'MAT-REC-', 'CC-', 'MAT-REC ', 'ALP-', 'ALPACA-', 'ALPACA ']:
    mask = df['codigo'].str.startswith(prefix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace('^' + prefix, '', regex=True)

df.tail(15)

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat
2117,55-25-18,VICUÑA 6020 2/15,KILOS,0.0,0.0,197.0,197.0,197.0,197.0,0.0,0.0,2/15,SM,
2118,55-25-19,CAFE OSC. 6034 2/15,KILOS,0.0,0.0,199.0,199.0,132.0,132.0,67.0,67.0,2/15,SM,
2119,55-25-21,VICUÑA CLARO 6006 2/15 SM,KILOS,0.0,0.0,186.0,186.0,186.0,186.0,0.0,0.0,2/15,SM,
2120,55-25-23,BURRITO 0017 2/15 SM,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/15,SM,
2121,55-25-25,VICUÑA 6011 2/15,KILOS,0.0,0.0,153.0,153.0,152.0,152.0,1.0,1.0,2/15,SM,
2122,57-26-01,VICUÑA 6107 2/15,KILOS,0.0,0.0,194.0,194.0,194.0,194.0,0.0,0.0,2/15,SM,
2123,57-26-02,NEGRO 5000 2/15,KILOS,0.0,0.0,198.0,198.0,198.0,198.0,0.0,0.0,2/15,SM,
2124,57-26-03,VICUÑA 6020 2/15 SM,KILOS,0.0,0.0,197.0,197.0,197.0,197.0,0.0,0.0,2/15,SM,
2125,57-26-04,CAFE OSC. 6015 2/15,KILOS,0.0,0.0,190.0,190.0,0.0,0.0,190.0,190.0,2/15,SM,
2126,57-26-05,NEGRO 5000 2/15,KILOS,0.0,0.0,185.0,185.0,184.0,184.0,1.0,1.0,2/15,SM,


verify that the cleaned `codigo` values are unique.

In [13]:
print(f"Unique codes: {df['codigo'].nunique()} out of {len(df)} rows")
print(f"Any nulls: {df['codigo'].isnull().any()}")

# Prefix distribution
codes = df['codigo'].astype(str)
print("\nSample codes:")
print(codes.head(10).to_list())
print(f"\nCodes with 'REC-': {codes.str.startswith('REC-').sum()}")
print(f"Codes with 'MAT-': {codes.str.startswith('MAT-').sum()}")
print(f"Codes with '-SM' suffix: {(codes.str.endswith('-SM').sum())}")
print(f"Codes with '-SM-1' suffix: {(codes.str.endswith('-SM-1').sum())}")
print(f"Codes with '-SM-N' suffix: {(codes.str.endswith('-SM-N').sum())}")
print(f"Codes with 'ALPACA' and '-SM' suffix: {(codes.str.endswith('ALPACA' and '-SM').sum())}")
print(f"Codes with '-SN' suffix: {(codes.str.endswith('-SN').sum())}")
print(f"Codes with '-SN-1' suffix: {(codes.str.endswith('-SN-1').sum())}")
print(f"Codes with '-CH' suffix: {(codes.str.endswith('-CH').sum())}")
print(f"Codes with '-N' suffix: {codes.str.endswith('-N').sum()}")
print(f"Codes with '-STOLL' suffix: {codes.str.endswith('-STOLL').sum()}")
print(f"Codes longer than 8 chars: {(codes.str.len() > 8).sum()}")
print(f"Codes longer than 11 chars: {(codes.str.len() > 11).sum()}")
print(codes[codes.str.len() > 8].unique())

Unique codes: 2055 out of 2132 rows
Any nulls: False

Sample codes:
['01-26-01', '01-26-05', '01-26-06', '01-26-11', '01-26-13', '01-26-16', '01-26-18', '01-26-19', '01-26-21', '01-26-23']

Codes with 'REC-': 0
Codes with 'MAT-': 1
Codes with '-SM' suffix: 0
Codes with '-SM-1' suffix: 1
Codes with '-SM-N' suffix: 0
Codes with 'ALPACA' and '-SM' suffix: 0
Codes with '-SN' suffix: 0
Codes with '-SN-1' suffix: 0
Codes with '-CH' suffix: 0
Codes with '-N' suffix: 0
Codes with '-STOLL' suffix: 0
Codes longer than 8 chars: 145
Codes longer than 11 chars: 5
<StringArray>
[  '09-32-26-01',   '09-32-26-02',   '09-32-26-03',   '09-32-26-04',
   '09-32-26-05',   '09-32-26-06',   '09-32-26-07',   '09-32-26-08',
   '09-32-26-09',   '09-32-26-11',
 ...
     '56-25-100',     '56-25-106',   '13-20-24-01',   '19-32-26-09',
   '19-32-26-16',   '19-32-26-24',     '23-26-114',     '23-26-121',
     '56-26-115', '19-24-05-SM-1']
Length: 145, dtype: str


In [14]:
with_words = df['codigo'].astype(str).str.contains(r'[A-Za-z]', regex=True, na=False)
count_words = with_words.sum()

if count_words > 24:
    df[with_words][['codigo']]
else:
    print(f"Son {count_words}, mayoria de muestras:")
    print(df[with_words][['codigo']].head(10))

Son 7, mayoria de muestras:
             codigo
1339      20-17-REC
1378      3/15-CONO
1379        MAT-REC
1401          ALM-1
1402          ALM-1
2101  19-24-05-SM-1
2102       AP-23-02


In [15]:
df['descripcion_material'] = ''
df.loc[df['codigo'].astype(str).str.startswith('-REC'), 'estado_mat'] = 'REC'
df.loc[df['codigo'].astype(str).str.endswith('MAT-REC'), 'estado_mat'] = 'MAT-REC'
df.loc[df['codigo'].astype(str).str.endswith('-CONO'), 'descripcion_material'] = 'CONO'
df.loc[df['codigo'].astype(str).str.startswith('ALM-'), 'descripcion_material'] = 'ALM'
df.loc[df['codigo'].astype(str).str.contains('-SM-'), 'tipo'] = 'SM'
df.loc[df['codigo'].astype(str).str.startswith('AP-'), 'tipo'] = 'AP'



for charsCode in ['MAT-REC']:
    mask = df['codigo'].str.endswith(charsCode)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(charsCode, 'N/A', regex=True)

for prefix in ['ALM-', 'AP-']:
    mask = df['codigo'].str.startswith(prefix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(prefix, '', regex=True)

for suffix in ['-CONO', '-REC']:
    mask = df['codigo'].str.endswith(suffix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(suffix + '$', '', regex=True)

for charSm in ['-SM']:
    mask = df['codigo'].str.contains(charSm)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(charSm, '', regex=True)

In [16]:
df['tipo'].unique()

<StringArray>
['HB', 'CICLAN', 'ALPACA / SM N', 'SM', 'STOLL', 'CH', 'M', 'ALPACA', 'AP']
Length: 9, dtype: str

In [17]:
with_words = df['codigo'].astype(str).str.contains(r'[A-Za-z]', regex=True, na=False)
count_words = with_words.sum()

if count_words > 24:
    df[with_words][['codigo']]
else:
    print(f"Son {count_words}, mayoria de muestras:")
    print(df[with_words][['codigo']].head(10))

Son 1, mayoria de muestras:
     codigo
1379    N/A


In [18]:
df.info()
print(f"sum: {df['codigo'].duplicated().sum()}")
df[df['codigo'].duplicated()]['codigo']

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   codigo                2132 non-null   str    
 1   descripcion           2132 non-null   str    
 2   unidad                2132 non-null   str    
 3   Saldoin-Cantidad      2132 non-null   float64
 4   Saldoin-Valor         2132 non-null   float64
 5   ent-Cantidad          2132 non-null   float64
 6   ent-Valor             2132 non-null   float64
 7   sal-Cantidad          2132 non-null   float64
 8   sal-Valor             2132 non-null   float64
 9   sadout-Cantidad       2132 non-null   float64
 10  saldout-Valor         2132 non-null   float64
 11  SUB-CATEGORIA         2132 non-null   str    
 12  tipo                  2132 non-null   str    
 13  estado_mat            2132 non-null   str    
 14  descripcion_material  2132 non-null   str    
dtypes: float64(8), str(7)
memory usa

1372        2/14
1378        3/15
1381       26-02
1382       26-04
1383       26-06
          ...   
2125    57-26-04
2127    57-26-06
2128    57-26-07
2129    57-26-08
2130    57-26-09
Name: codigo, Length: 78, dtype: str

export the dataframe of duplicated `codigo` values for further analysis.

In [19]:
# Filtrar todos los duplicados (incluyendo la primera aparición)
duplicados = df[df['codigo'].duplicated(keep=False)]

# Resetear el índice para que aparezca como columna en el CSV
duplicados = duplicados.reset_index()

# Exportar a CSV
duplicados.to_csv('duplicados_detalle.csv', index=False)


In [20]:
df[df['codigo'].astype(str).str.contains('20-26-11')]

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material
617,20-26-11,KANTUTA 3045 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18,HB,,
1556,20-26-11,PACEÑITA 2010 2/13-N,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/13 N,ALPACA / SM N,,


In [21]:
print("Valores unicos en tipo")
df['tipo'].unique()


Valores unicos en tipo


<StringArray>
['HB', 'CICLAN', 'ALPACA / SM N', 'SM', 'STOLL', 'CH', 'M', 'ALPACA', 'AP']
Length: 9, dtype: str

In [22]:

print("Valores unicos en estado_mat")
df['estado_mat'].unique()


Valores unicos en estado_mat


<StringArray>
['', 'REC', 'MAT-REC']
Length: 3, dtype: str

In [23]:
print("Valores unicos en descripcion_material")
df['descripcion_material'].unique()

Valores unicos en descripcion_material


<StringArray>
['', 'CONO', 'ALM']
Length: 3, dtype: str

In [24]:
df.to_csv('../../reference/warehouse/cleaned/df-code-cleeaning.csv', index=False)

---
## 3. Column B — descripcion (main focus)

String to normalize. Dominant pattern:
```
COLOR_NAME  COLOR_CODE  THICKNESS
```
Separated by double spaces. Examples:
- `KANTUTA  3045  2/18` → name=KANTUTA, code=3045, thickness=2/18
- `AM. LIMON  0024  2/18` → name=AM. LIMON, code=0024, thickness=2/18
- `AZUL PETROLEO  7009  2/18`

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
dframe = pd.read_csv('../../reference/warehouse/cleaned/df-code-cleeaning.csv')
dframe.head()

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material
0,01-26-01,KANTUTA 3045 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18,HB,NaN,NaN
1,01-26-05,AM. LIMON 0024 2/18,KILOS,0.0,0.0,201.5,201.5,201.5,201.5,0.0,0.0,2/18,HB,NaN,NaN
2,01-26-06,AZUL PETROLEO 7009 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18,HB,NaN,NaN
3,01-26-11,BLANCO 0010 2/18,KILOS,0.0,0.0,202.5,202.5,202.5,202.5,0.0,0.0,2/18,HB,NaN,NaN
4,01-26-13,BLANCO 0010 2/18,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/18,HB,NaN,NaN


In [3]:
descs = dframe['descripcion'].astype(str)
print(f"Unique descriptions: {descs.nunique()} out of {len(descs)}")
print(f"Nulls: {descs.isnull().sum()}")

Unique descriptions: 706 out of 2132
Nulls: 0


In [4]:
mask_desc = dframe.apply(lambda row: str(row['SUB-CATEGORIA']) in str(row['descripcion']), axis=1)

dframe['titulo'] = ''

dframe.loc[mask_desc, 'titulo'] = dframe.loc[mask_desc, 'SUB-CATEGORIA']
dframe.loc[mask_desc, 'descripcion'] = dframe.loc[mask_desc].apply(
    lambda row: str(row['descripcion']).replace(str(row['SUB-CATEGORIA']), '').strip(), axis=1
)

print(f"Match: {mask_desc.sum()} | No Match: {(~mask_desc).sum()}")

Match: 1811 | No Match: 321


In [5]:
# Donde titulo esta vacio, buscar patron X/XX en descripcion
without_yarn_mask = dframe['titulo'] == ''

# Extraer el primer X/XX que aparezca
thickness_found = dframe.loc[without_yarn_mask, 'descripcion'].str.extract(r'(\d+/\d+)', expand=False)

# Asignar a titulo y eliminar de descripcion
dframe.loc[without_yarn_mask & thickness_found.notna(), 'titulo'] = thickness_found
dframe.loc[without_yarn_mask & thickness_found.notna(), 'descripcion'] = (
    dframe.loc[without_yarn_mask & thickness_found.notna(), 'descripcion']
    .str.replace(r'\s*\d+/\d+\s*', ' ', regex=True)
    .str.strip()
)

print(f"Con X/XX: {thickness_found.notna().sum()} | Sin X/XX: {thickness_found.isna().sum()}")
print(f"Total con titulo: {(dframe['titulo'] != '').sum()} / {len(dframe)}")
print(f"Aun sin titulo: {(dframe['titulo'] == '').sum()} / {len(dframe)}")

Con X/XX: 305 | Sin X/XX: 16
Total con titulo: 2116 / 2132
Aun sin titulo: 16 / 2132


In [6]:
print("Valores unicos en titulo")
dframe.groupby(['titulo']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

Valores unicos en titulo


,titulo,counts
10,2/18,875
17,2/32,472
4,2/12,256
5,2/13,199
19,4/9,83
2,2/10,60
21,STOLL,46
18,3/15,28
3,2/11,24
7,2/15,19


In [7]:
dframe.loc[dframe['titulo'] == 'RECUPERADO']

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
1339,20-17,SAL ANT PROD TERM,KILOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RECUPERADO,HB,NaN,NaN,RECUPERADO
1367,2/10,MATERIAL 2/10,KILOS,26.0,26.0,522.0,522.0,170.0,170.0,378.0,378.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1368,2/11,MATERIAL 2/11 SM,KILOS,58.0,58.0,57.0,57.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1369,2/12,MATERIAL 2/12,KILOS,82.0,82.0,1649.0,1649.0,395.0,395.0,1336.0,1336.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1370,2/13,MATERIAL 2/13,KILOS,542.0,542.0,1055.0,1055.0,0.0,0.0,1597.0,1597.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1371,2/14,MATERIAL 2/14,KILOS,9.0,9.0,0.0,0.0,0.0,0.0,9.0,9.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1372,2/14,MATERIAL 2/14 STOLL,KILOS,228.0,228.0,0.0,0.0,9.0,9.0,219.0,219.0,RECUPERADO,STOLL,MAT-REC,NaN,RECUPERADO
1373,2/18,MATERIAL 2/18,KILOS,307.0,307.0,1914.0,1914.0,1945.0,1945.0,276.0,276.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1374,2/28,MATERIAL 2/28,KILOS,115.0,115.0,0.0,0.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1375,2/30,MATERIAL 2/30,KILOS,77.0,77.0,3.0,3.0,0.0,0.0,80.0,80.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO


In [ ]:
rec_yarn_mask = dframe['titulo'] == 'RECUPERADO'

statmat_found = dframe.loc[rec_yarn_mask, 'descripcion'].str.extract(r'(\d+/\d+)', expand=False)

dframe.loc[rec_yarn_mask & statmat_found.notna(), 'titulo'] = statmat_found

dframe.loc[rec_yarn_mask & statmat_found.notna(), 'descripcion'] = (
    dframe.loc[rec_yarn_mask & statmat_found.notna(), 'descripcion']
    .str.replace(r'\s*\d+/\d+\s*', ' ', regex=True)
    .str.strip()
)

dframe.loc[rec_yarn_mask, 'descripcion'] = (
    dframe.loc[rec_yarn_mask, 'descripcion']
    .str.replace(r'MATERIAL', ' ', regex=True)
    .str.strip()
)

dframe.loc[rec_yarn_mask]

In [9]:
dframe[1367:1370]

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
1367,2/10,MATERIAL,KILOS,26.0,26.0,522.0,522.0,170.0,170.0,378.0,378.0,RECUPERADO,HB,MAT-REC,NaN,2/10
1368,2/11,MATERIAL SM,KILOS,58.0,58.0,57.0,57.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,2/11
1369,2/12,MATERIAL,KILOS,82.0,82.0,1649.0,1649.0,395.0,395.0,1336.0,1336.0,RECUPERADO,HB,MAT-REC,NaN,2/12


In [10]:
dframe.loc[rec_yarn_mask]

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
1339,20-17,SAL ANT PROD TERM,KILOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RECUPERADO,HB,NaN,NaN,RECUPERADO
1367,2/10,MATERIAL,KILOS,26.0,26.0,522.0,522.0,170.0,170.0,378.0,378.0,RECUPERADO,HB,MAT-REC,NaN,2/10
1368,2/11,MATERIAL SM,KILOS,58.0,58.0,57.0,57.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,2/11
1369,2/12,MATERIAL,KILOS,82.0,82.0,1649.0,1649.0,395.0,395.0,1336.0,1336.0,RECUPERADO,HB,MAT-REC,NaN,2/12
1370,2/13,MATERIAL,KILOS,542.0,542.0,1055.0,1055.0,0.0,0.0,1597.0,1597.0,RECUPERADO,HB,MAT-REC,NaN,2/13
1371,2/14,MATERIAL,KILOS,9.0,9.0,0.0,0.0,0.0,0.0,9.0,9.0,RECUPERADO,HB,MAT-REC,NaN,2/14
1372,2/14,MATERIAL STOLL,KILOS,228.0,228.0,0.0,0.0,9.0,9.0,219.0,219.0,RECUPERADO,STOLL,MAT-REC,NaN,2/14
1373,2/18,MATERIAL,KILOS,307.0,307.0,1914.0,1914.0,1945.0,1945.0,276.0,276.0,RECUPERADO,HB,MAT-REC,NaN,2/18
1374,2/28,MATERIAL,KILOS,115.0,115.0,0.0,0.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,2/28
1375,2/30,MATERIAL,KILOS,77.0,77.0,3.0,3.0,0.0,0.0,80.0,80.0,RECUPERADO,HB,MAT-REC,NaN,2/30


In [11]:
dframe.loc[dframe['titulo'] == 'STOLL'].head(15)

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
2051,01-26-29,BLANCO 0010,KILOS,0.0,0.0,192.0,192.0,192.0,192.0,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2052,02-26-53,NEGRO 5000,KILOS,0.0,0.0,206.4,206.4,206.4,206.4,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2053,03-26-10,AZUL 7052,KILOS,0.0,0.0,208.8,208.8,208.8,208.8,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2054,03-26-22,AZUL 7052,KILOS,0.0,0.0,200.0,200.0,200.0,200.0,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2055,05-26-13,VERDE OSC. 4016,KILOS,0.0,0.0,161.0,161.0,0.0,0.0,161.0,161.0,STOLL,HB,NaN,NaN,STOLL
2056,05-26-27,VERDE OSC. 4016,KILOS,0.0,0.0,154.0,154.0,154.0,154.0,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2057,05-26-36,AZUL 7052,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2058,05-26-41,NEGRO 5000,KILOS,0.0,0.0,184.0,184.0,184.0,184.0,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2059,07-26-10,NEGRO 5000,KILOS,0.0,0.0,200.0,200.0,200.0,200.0,0.0,0.0,STOLL,HB,NaN,NaN,STOLL
2060,07-26-15,API 7033,KILOS,0.0,0.0,195.0,195.0,0.0,0.0,195.0,195.0,STOLL,HB,NaN,NaN,STOLL


In [15]:
stoll_yarn_mask = dframe['titulo'] == 'STOLL'
print(stoll_yarn_mask.unique())

[False  True]


In [17]:
stoll_yarn_mask = dframe['titulo'] == 'STOLL'

stoll_found = dframe.loc[stoll_yarn_mask, 'titulo'].str.extract(r'(STOLL)', expand=False)

dframe.loc[stoll_yarn_mask & stoll_found.notna(), 'tipo'] = stoll_found

dframe.loc[stoll_yarn_mask & stoll_found.notna(), 'titulo'] = (
    dframe.loc[stoll_yarn_mask & stoll_found.notna(), 'titulo']
    .str.replace(r'STOLL', ' ', regex=True)
    .str.strip()
)

dframe.loc[stoll_yarn_mask]

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
2051,01-26-29,BLANCO 0010,KILOS,0.0,0.0,192.0,192.0,192.0,192.0,0.0,0.0,STOLL,STOLL,NaN,NaN,
2052,02-26-53,NEGRO 5000,KILOS,0.0,0.0,206.4,206.4,206.4,206.4,0.0,0.0,STOLL,STOLL,NaN,NaN,
2053,03-26-10,AZUL 7052,KILOS,0.0,0.0,208.8,208.8,208.8,208.8,0.0,0.0,STOLL,STOLL,NaN,NaN,
2054,03-26-22,AZUL 7052,KILOS,0.0,0.0,200.0,200.0,200.0,200.0,0.0,0.0,STOLL,STOLL,NaN,NaN,
2055,05-26-13,VERDE OSC. 4016,KILOS,0.0,0.0,161.0,161.0,0.0,0.0,161.0,161.0,STOLL,STOLL,NaN,NaN,
2056,05-26-27,VERDE OSC. 4016,KILOS,0.0,0.0,154.0,154.0,154.0,154.0,0.0,0.0,STOLL,STOLL,NaN,NaN,
2057,05-26-36,AZUL 7052,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,STOLL,STOLL,NaN,NaN,
2058,05-26-41,NEGRO 5000,KILOS,0.0,0.0,184.0,184.0,184.0,184.0,0.0,0.0,STOLL,STOLL,NaN,NaN,
2059,07-26-10,NEGRO 5000,KILOS,0.0,0.0,200.0,200.0,200.0,200.0,0.0,0.0,STOLL,STOLL,NaN,NaN,
2060,07-26-15,API 7033,KILOS,0.0,0.0,195.0,195.0,0.0,0.0,195.0,195.0,STOLL,STOLL,NaN,NaN,


In [18]:
dframe.groupby(['titulo']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

,titulo,counts
10,2/18,876
17,2/32,473
4,2/12,257
5,2/13,200
19,4/9,83
0,,62
2,2/10,61
18,3/15,30
3,2/11,25
7,2/15,19


In [26]:
dframe.loc[dframe['titulo'] == 'RECUPERADO'].head()

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
1339,20-17,SAL ANT PROD TERM,KILOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RECUPERADO,HB,NaN,NaN,RECUPERADO
1379,NaN,STOLL,KILOS,46.0,46.0,780.3,780.3,8.0,8.0,818.3,818.3,RECUPERADO,STOLL,MAT-REC,NaN,RECUPERADO


In [23]:
dframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   codigo                2131 non-null   str    
 1   descripcion           2132 non-null   str    
 2   unidad                2132 non-null   str    
 3   Saldoin-Cantidad      2132 non-null   float64
 4   Saldoin-Valor         2132 non-null   float64
 5   ent-Cantidad          2132 non-null   float64
 6   ent-Valor             2132 non-null   float64
 7   sal-Cantidad          2132 non-null   float64
 8   sal-Valor             2132 non-null   float64
 9   sadout-Cantidad       2132 non-null   float64
 10  saldout-Valor         2132 non-null   float64
 11  SUB-CATEGORIA         2132 non-null   str    
 12  tipo                  2132 non-null   str    
 13  estado_mat            38 non-null     str    
 14  descripcion_material  3 non-null      str    
 15  titulo                2132 non-n

In [27]:
# Crear una máscara booleana: True si hay al menos un dígito en la descripción
mask_numbers = dframe['descripcion'].str.contains(r'\d', regex=True)

# Contar cuántas filas cumplen la condición
count_with_numbers = mask_numbers.sum()

# Contar cuántas no tienen números
count_without_numbers = (~mask_numbers).sum()

print(f"Con números: {count_with_numbers} | Sin números: {count_without_numbers}")


Con números: 2103 | Sin números: 29


In [28]:
dframe.loc[~mask_numbers]

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
1092,20-17-2/32,SAL ANT PROD TERM,KILOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2/32,HB,NaN,NaN,2/32
1339,20-17,SAL ANT PROD TERM,KILOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RECUPERADO,HB,NaN,NaN,RECUPERADO
1352,25-06-25-22,GRISS (REMATE),KILOS,11.0,11.0,0.0,0.0,0.0,0.0,11.0,11.0,RECUPERADO,HB,NaN,NaN,
1367,2/10,,KILOS,26.0,26.0,522.0,522.0,170.0,170.0,378.0,378.0,RECUPERADO,HB,MAT-REC,NaN,2/10
1368,2/11,SM,KILOS,58.0,58.0,57.0,57.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,2/11
1369,2/12,,KILOS,82.0,82.0,1649.0,1649.0,395.0,395.0,1336.0,1336.0,RECUPERADO,HB,MAT-REC,NaN,2/12
1370,2/13,,KILOS,542.0,542.0,1055.0,1055.0,0.0,0.0,1597.0,1597.0,RECUPERADO,HB,MAT-REC,NaN,2/13
1371,2/14,,KILOS,9.0,9.0,0.0,0.0,0.0,0.0,9.0,9.0,RECUPERADO,HB,MAT-REC,NaN,2/14
1372,2/14,STOLL,KILOS,228.0,228.0,0.0,0.0,9.0,9.0,219.0,219.0,RECUPERADO,STOLL,MAT-REC,NaN,2/14
1373,2/18,,KILOS,307.0,307.0,1914.0,1914.0,1945.0,1945.0,276.0,276.0,RECUPERADO,HB,MAT-REC,NaN,2/18


In [31]:
dframe.to_csv('../../reference/warehouse/cleaned/df-yarn-cleaning.csv', index=False)

In [35]:
print(dframe.loc[(dframe['titulo'] == 'RECUPERADO') & (dframe['estado_mat'] == 'MAT-REC'), ['codigo', 'estado_mat']])

     codigo estado_mat
1379    NaN    MAT-REC


### 3.1 Colour Cases

In [32]:
dfcolor = pd.read_csv('../../reference/warehouse/cleaned/df-yarn-cleaning.csv')
dfcolor.head()

,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
0,01-26-01,KANTUTA 3045,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18,HB,NaN,NaN,2/18
1,01-26-05,AM. LIMON 0024,KILOS,0.0,0.0,201.5,201.5,201.5,201.5,0.0,0.0,2/18,HB,NaN,NaN,2/18
2,01-26-06,AZUL PETROLEO 7009,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18,HB,NaN,NaN,2/18
3,01-26-11,BLANCO 0010,KILOS,0.0,0.0,202.5,202.5,202.5,202.5,0.0,0.0,2/18,HB,NaN,NaN,2/18
4,01-26-13,BLANCO 0010,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/18,HB,NaN,NaN,2/18


---
## 6. Next Steps

- [ ] Refine parsing for complex compound names (4+ words)
- [ ] Handle extra qualifiers (STOLL, TIPO N, -D, MANCH)
- [ ] Code typo detection (e.g., CANCARIO vs CANARIO)
- [ ] Validate thickness consistency between Column B and SUB-CATEGORIA
- [ ] Export cleaned dataframe to CSV / reference data